In [ ]:
# ============================================================
# 🛍️ SHOPPING RESEARCH MULTI-AGENT SYSTEM
# LangGraph + Groq + Gradio
# Google Colab - One Cell
# ============================================================


# ============================================================
# 1️⃣ INSTALL REQUIRED LIBRARIES
# ============================================================

!pip -q install langgraph langchain-groq gradio


# ============================================================
# 2️⃣ IMPORT LIBRARIES
# ============================================================

import os
from typing import TypedDict

import gradio as gr

from getpass import getpass

from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END


# ============================================================
# 3️⃣ ENTER YOUR GROQ API KEY
# ============================================================

print("🔑 Enter your Groq API Key:")

GROQ_API_KEY = getpass("Groq API Key: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY


# ============================================================
# 4️⃣ CREATE THE AI MODEL
# ============================================================

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)


# ============================================================
# 5️⃣ CREATE SHARED STATE
# ============================================================

class ShoppingState(TypedDict):

    product: str
    budget: str
    requirements: str

    research: str
    comparison: str
    recommendation: str


# ============================================================
# 6️⃣ PRODUCT RESEARCH AGENT
# ============================================================

def research_agent(state):

    product = state["product"]
    budget = state["budget"]
    requirements = state["requirements"]

    prompt = f"""
You are a Shopping Research Agent.

The user wants to buy:

Product:
{product}

Budget:
{budget}

Requirements:
{requirements}

Research the product category and provide useful
shopping information.

Include:

1. What the product is
2. Important specifications to look for
3. Best features to consider
4. Advantages
5. Disadvantages
6. Suitable brands or product types
7. Things a buyer should check before purchasing
8. Value-for-money considerations

Do NOT invent exact current prices or availability.

Explain everything in simple English.
"""

    response = llm.invoke(prompt)

    print("\n🔎 RESEARCH AGENT FINISHED")

    return {
        "research": response.content
    }


# ============================================================
# 7️⃣ COMPARISON AGENT
# ============================================================

def comparison_agent(state):

    research = state["research"]

    product = state["product"]
    budget = state["budget"]

    prompt = f"""
You are a Product Comparison Agent.

The user wants:

Product:
{product}

Budget:
{budget}

Here is the research:

{research}

Create a useful comparison of possible product choices.

Compare products/categories using:

- Performance
- Features
- Quality
- Reliability
- Ease of use
- Value for money
- Advantages
- Disadvantages
- Best use case

If exact product models or current prices are not available,
do not invent them.

Instead, explain what type of product would be the
best choice for the user's requirements.

Use a clear table when useful.
"""

    response = llm.invoke(prompt)

    print("⚖️ COMPARISON AGENT FINISHED")

    return {
        "comparison": response.content
    }


# ============================================================
# 8️⃣ RECOMMENDATION AGENT
# ============================================================

def recommendation_agent(state):

    product = state["product"]
    budget = state["budget"]
    requirements = state["requirements"]

    research = state["research"]
    comparison = state["comparison"]

    prompt = f"""
You are a Shopping Recommendation Agent.

User requirements:

Product:
{product}

Budget:
{budget}

Requirements:
{requirements}

RESEARCH:
{research}

COMPARISON:
{comparison}

Based on all the information, give the user
a final shopping recommendation.

Include:

1. 🏆 Best overall choice
2. 💰 Best value-for-money choice
3. ⭐ Best choice for the user's requirements
4. ⚠️ What to avoid
5. 🛒 Final buying checklist

Give a clear final recommendation.

Do not invent current prices, availability,
ratings or product specifications.

Use simple English and make the answer
easy for a beginner to understand.
"""

    response = llm.invoke(prompt)

    print("🛍️ RECOMMENDATION AGENT FINISHED")

    return {
        "recommendation": response.content
    }


# ============================================================
# 9️⃣ CREATE LANGGRAPH
# ============================================================

graph = StateGraph(ShoppingState)


# ============================================================
# 🔟 ADD AGENTS AS NODES
# ============================================================

graph.add_node(
    "researcher",
    research_agent
)

graph.add_node(
    "comparator",
    comparison_agent
)

graph.add_node(
    "recommender",
    recommendation_agent
)


# ============================================================
# 1️⃣1️⃣ CONNECT THE AGENTS
# ============================================================

graph.add_edge(
    START,
    "researcher"
)

graph.add_edge(
    "researcher",
    "comparator"
)

graph.add_edge(
    "comparator",
    "recommender"
)

graph.add_edge(
    "recommender",
    END
)


# ============================================================
# 1️⃣2️⃣ COMPILE THE GRAPH
# ============================================================

app = graph.compile()


# ============================================================
# 1️⃣3️⃣ SHOPPING RESEARCH FUNCTION
# ============================================================

def shopping_research(product, budget, requirements):

    if not product:
        return (
            "⚠️ Please enter a product.",
            "",
            ""
        )

    if not budget:
        budget = "No specific budget"

    if not requirements:
        requirements = "No additional requirements"

    result = app.invoke({

        "product": product,

        "budget": budget,

        "requirements": requirements,

        "research": "",

        "comparison": "",

        "recommendation": ""
    })

    return (
        result["research"],
        result["comparison"],
        result["recommendation"]
    )


# ============================================================
# 1️⃣4️⃣ CUSTOM GRADIO CSS
# ============================================================

custom_css = """

/* Main background */

.gradio-container {
    background:
        linear-gradient(
            135deg,
            #0f172a 0%,
            #172554 45%,
            #312e81 100%
        ) !important;

    color: white !important;
}


/* Main title */

.title-box {
    text-align: center;

    padding: 25px;

    border-radius: 20px;

    background:
        linear-gradient(
            135deg,
            #7c3aed,
            #2563eb
        );

    box-shadow:
        0 10px 30px rgba(0,0,0,0.35);

    margin-bottom: 20px;
}


/* Cards */

.agent-card {

    border-radius: 18px !important;

    padding: 20px !important;

    background:
        rgba(255,255,255,0.08) !important;

    border:
        1px solid rgba(255,255,255,0.15) !important;

    box-shadow:
        0 8px 25px rgba(0,0,0,0.25);
}


/* Input boxes */

textarea,
input {

    border-radius: 12px !important;

    border:
        1px solid #6366f1 !important;
}


/* Button */

#research-button {

    background:
        linear-gradient(
            90deg,
            #f97316,
            #ec4899,
            #8b5cf6
        ) !important;

    color: white !important;

    font-size: 18px !important;

    font-weight: bold !important;

    border-radius: 14px !important;

    padding: 12px !important;

    border: none !important;
}


/* Output boxes */

.output-box {

    border-radius: 15px !important;

    border:
        1px solid rgba(255,255,255,0.15) !important;
}


/* Agent headings */

.agent-heading {

    font-size: 20px;

    font-weight: bold;

    padding: 10px;

    border-radius: 10px;

    text-align: center;

    margin-bottom: 10px;
}

"""


# ============================================================
# 1️⃣5️⃣ CREATE GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    css=custom_css,
    title="AI Shopping Research Agent"
) as demo:


    # ========================================================
    # HEADER
    # ========================================================

    gr.HTML(
        """
        <div class="title-box">

            <h1 style="
                font-size: 38px;
                margin-bottom: 8px;
            ">
                🛍️ AI Shopping Research Agent
            </h1>

            <p style="
                font-size: 18px;
                margin: 0;
            ">
                Multi-Agent AI System for Smarter Shopping Decisions
            </p>

        </div>
        """
    )


    # ========================================================
    # INPUT SECTION
    # ========================================================

    with gr.Row():

        with gr.Column(
            scale=1,
            elem_classes="agent-card"
        ):

            gr.Markdown(
                """
                ## 🛒 Shopping Requirements

                Tell the AI what you want to buy.
                """
            )

            product_input = gr.Textbox(
                label="📦 Product",
                placeholder="Example: Laptop",
                lines=1
            )

            budget_input = gr.Textbox(
                label="💰 Budget",
                placeholder="Example: ₹60,000",
                lines=1
            )

            requirements_input = gr.Textbox(
                label="⭐ Requirements",
                placeholder=(
                    "Example: "
                    "Good for programming, AI/ML, "
                    "16GB RAM, good battery life"
                ),
                lines=4
            )

            research_button = gr.Button(
                "🚀 START SHOPPING RESEARCH",
                variant="primary",
                elem_id="research-button"
            )


    # ========================================================
    # OUTPUT SECTION
    # ========================================================

    gr.Markdown(
        """
        # 🤖 AI Agent Results

        The agents work sequentially:

        **Research → Comparison → Recommendation**
        """
    )


    # ========================================================
    # RESEARCH OUTPUT
    # ========================================================

    with gr.Group(
        elem_classes="agent-card"
    ):

        gr.HTML(
            """
            <div class="agent-heading"
                 style="
                 background:#2563eb;
                 color:white;
                 ">
                🔎 RESEARCH AGENT
            </div>
            """
        )

        research_output = gr.Markdown(
            elem_classes="output-box"
        )


    # ========================================================
    # COMPARISON OUTPUT
    # ========================================================

    with gr.Group(
        elem_classes="agent-card"
    ):

        gr.HTML(
            """
            <div class="agent-heading"
                 style="
                 background:#9333ea;
                 color:white;
                 ">
                ⚖️ COMPARISON AGENT
            </div>
            """
        )

        comparison_output = gr.Markdown(
            elem_classes="output-box"
        )


    # ========================================================
    # RECOMMENDATION OUTPUT
    # ========================================================

    with gr.Group(
        elem_classes="agent-card"
    ):

        gr.HTML(
            """
            <div class="agent-heading"
                 style="
                 background:#db2777;
                 color:white;
                 ">
                🏆 RECOMMENDATION AGENT
            </div>
            """
        )

        recommendation_output = gr.Markdown(
            elem_classes="output-box"
        )


    # ========================================================
    # FOOTER
    # ========================================================

    gr.HTML(
        """
        <div style="
            text-align:center;
            padding:20px;
            margin-top:20px;
            opacity:0.8;
        ">

            <p>
                🤖 Powered by LangGraph + Groq + Gradio
            </p>

            <p>
                🔎 Research • ⚖️ Compare • 🏆 Recommend
            </p>

        </div>
        """
    )


    # ========================================================
    # BUTTON ACTION
    # ========================================================

    research_button.click(

        fn=shopping_research,

        inputs=[
            product_input,
            budget_input,
            requirements_input
        ],

        outputs=[
            research_output,
            comparison_output,
            recommendation_output
        ]
    )


# ============================================================
# 1️⃣6️⃣ LAUNCH GRADIO
# ============================================================

print("\n")
print("=" * 70)
print("🛍️ AI SHOPPING RESEARCH AGENT")
print("=" * 70)
print("✅ LangGraph Multi-Agent System Ready")
print("🔎 Research Agent")
print("⚖️ Comparison Agent")
print("🏆 Recommendation Agent")
print("=" * 70)


demo.launch(
    share=True,
    debug=True
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.6 MB/s eta 0:00:00
🔑 Enter your Groq API Key:
Groq API Key: ··········


/tmp/ipykernel_999/3936115857.py:475: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(




🛍️ AI SHOPPING RESEARCH AGENT
✅ LangGraph Multi-Agent System Ready
🔎 Research Agent
⚖️ Comparison Agent
🏆 Recommendation Agent
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c1ed9bb5064c3c0513.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🔎 RESEARCH AGENT FINISHED
⚖️ COMPARISON AGENT FINISHED
🛍️ RECOMMENDATION AGENT FINISHED
